In [1]:
# req libs
from rerankers import mmr, steck_2018_re_rank, xquad, ugf, rerank_all, candidate_pool, proxy_quality
import numpy as np
from metrics import ild, miscalibration, aplt, gru


### Task 3.1 - Rerankers

Every reranker in [rerankers.py](rerankers.py) has the same interface:

```
rerank(scores_u, candidates_u, k, lam, **context) -> k item ids, best first
```

* `scores_u`: the base model's scores for one user, indexed by item id (one row of a score matrix).
* `candidates_u`: the item ids the reranker may choose from (the user's top-N pool).
* `lam` in [0, 1]: 0 returns the base ranking, 1 puts all weight on the beyond-accuracy objective.
* `context`: what the method needs, e.g. `genres` for MMR.

The sanity checks below run on the toy catalogue from [task_2_1_beyond_accuracy.ipynb](task_2_1_beyond_accuracy.ipynb).

In [ ]:
# catalogue from task_2_1_beyond_accuracy
G = np.array([[1, 0, 0], [1, 0, 0], [0, 1, 0], [1, 1, 0], [0, 0, 1], [1, 1, 1]])
scores = np.array([0.9, 0.8, 0.3, 0.2, 0.5, 0.1]) 
cands = np.arange(6)
subset = np.array([1, 2, 4, 5])                      #  pool leaves out top item
LAMBDAS = [0, 0.25, 0.5, 0.75, 1]


def check_basics(rerank, k=3, **context):
    """Checks every reranker must pass: length, no duplicates, only candidates, deterministic,
    and λ = 0 returns the base ranking of the pool."""
    for pool in (cands, subset):
        for lam in LAMBDAS:
            out = list(rerank(scores, pool, k=k, lam=lam, **context))
            assert len(out) == k and len(set(out)) == k, f"λ={lam}: wrong length or duplicates {out}"
            assert set(out) <= set(pool.tolist()), f"λ={lam}: returned a non-candidate {out}"
            assert out == list(rerank(scores, pool, k=k, lam=lam, **context)), f"λ={lam}: not deterministic"
        base = pool[np.argsort(-scores[pool], kind="stable")][:k].tolist()
        assert list(rerank(scores, pool, k=k, lam=0, **context)) == base, "λ = 0 changed the base ranking"

    try:
        rerank(scores, cands[:2], k=3, lam=0.5, **context)
    except ValueError:
        pass
    else:
        raise AssertionError("k larger than the pool should raise ValueError")

In [ ]:
# MMR
check_basics(mmr, genres=G)

assert list(mmr(scores, cands, k=2, lam=0.9, genres=G)) == [0, 4]


assert list(mmr(scores, cands, k=3, lam=0.9, genres=G)) == [0, 4, 2]

assert np.isclose(ild([mmr(scores, cands, k=3, lam=0, genres=G)], G)[0], 2 / 3)
assert np.isclose(ild([mmr(scores, cands, k=3, lam=0.9, genres=G)], G)[0], 1)
print("MMR toy checks OK")

In [ ]:
# Steck
HIST = [2, 2, 4]                                     # user watched ONLY B and C 
check_basics(steck_2018_re_rank, genres=G, history=HIST)

mc = {lam: miscalibration([steck_2018_re_rank(scores, cands, k=3, lam=lam, genres=G, history=HIST)], [HIST], G)[0]
      for lam in LAMBDAS}
assert mc[1] <= mc[0], mc
assert all(a >= b - 1e-12 for a, b in zip(list(mc.values()), list(mc.values())[1:])), f"MC_KL not monotone in λ: {mc}"
print("MC_KL by λ:", {lam: round(v, 3) for lam, v in mc.items()})
print("Steck toy checks OK")

In [ ]:
# xQuAD
HEAD = np.array([1, 1, 0, 0, 1, 0], bool)           
for smooth in (False, True):
    check_basics(xquad, k=4, short_head=HEAD, smooth=smooth)                    # uniform 
    check_basics(xquad, k=4, short_head=HEAD, history=[0, 2], smooth=smooth)    # personalised

assert xquad(scores, cands, k=4, lam=0, short_head=HEAD) == [0, 1, 4, 2]
assert xquad(scores, cands, k=4, lam=1, short_head=HEAD)[:2] == [0, 2]

assert not HEAD[xquad(scores, cands, k=4, lam=1, short_head=HEAD, history=[2, 3, 5])[0]]

for hist, want in (([0, 2], 0.5), ([0, 1, 4, 2], 0.75)):
    out = xquad(scores, cands, k=4, lam=1, short_head=HEAD, history=hist, smooth=True)
    assert HEAD[out].mean() == want, (hist, out)

tail = ~HEAD
assert aplt([xquad(scores, cands, k=4, lam=0, short_head=HEAD)], tail)[0] == 0.25
assert aplt([xquad(scores, cands, k=4, lam=1, short_head=HEAD, smooth=True)], tail)[0] == 0.5
print("xQuAD toy checks OK")

UGF couples users through the group constraint, so it does not fit the per-user interface: `ugf(scores, users, k, eps, in_group_1)` takes the whole score matrix and solves one ILP. `eps` replaces λ, inverted: a large `eps` gives the base top-K. The quality used in the constraint is the mean min-max normalised score of the list (`proxy_quality`), since test labels can't be used; the real gap is measured on test with `gru` on NDCG@10.

In [ ]:
# UGF
rng = np.random.default_rng(0)
S = rng.random((200, 300))
users = np.arange(200)
g1 = rng.random(200) < 0.3
S[g1] = S[g1] ** 0.5                                 # skews g1 scores to top of the pool
S[:, 0] = -np.inf                                    # padding 
S[:, 1:6] = -np.inf                                  # "seen" items
K, N = 10, 50

base = np.array([candidate_pool(S[u], N)[:K] for u in users])
assert (rerank_all(mmr, S, users, K, lam=0, n_candidates=N, genres=rng.random((300, 4)) < 0.5) == base).all()

gap0 = gru(proxy_quality(S, users, base, N), g1)
assert (ugf(S, users, K, eps=1.0, in_group_1=g1, n_candidates=N) == base).all(), "large eps changed the base top-K"

gaps, totals = [], []
for eps in (gap0 / 2, gap0 / 4, gap0 / 10):
    out = ugf(S, users, K, eps=eps, in_group_1=g1, n_candidates=N)
    assert all(len(set(r)) == K for r in out) and all(set(r) <= set(candidate_pool(S[u], N)) for u, r in zip(users, out))
    q = proxy_quality(S, users, out, N)
    gaps.append(gru(q, g1))
    totals.append(q.sum())
    assert gaps[-1] <= eps + 1e-6, (eps, gaps[-1])
assert all(a >= b for a, b in zip(gaps, gaps[1:])), f"gap did not shrink with eps: {gaps}"
assert all(a >= b for a, b in zip(totals, totals[1:])), "a tighter eps should not increase total quality"
print(f"proxy gap: base {gap0:.4f} -> " + ", ".join(f"{g:.4f}" for g in gaps))
print("UGF toy checks OK")